# WoW Economy Forecaster -- Model Development

This notebook explores the feature engineering pipeline, model architecture, and forecast generation that power the WoW Economy Forecaster. It builds on the data intuition from Notebook 01 (EDA) and sets the stage for formal evaluation in Notebook 03 (Backtest).

## Architecture Summary

The forecaster trains **one global LightGBM model per forecast horizon** (1d, 7d, 28d) across all archetype-realm series. This cross-archetype training strategy is deliberate:

- It enables the model to learn *shared* patterns like "consumables spike before major events" across all consumable archetypes.
- Cold-start archetypes with limited data benefit from knowledge transfer through shared features.
- `archetype_id` is excluded as a feature to prevent memorizing TWW-specific IDs (enabling transfer to Midnight).

The model uses MAE (L1) loss rather than MSE -- WoW prices have outliers from market manipulation and low-volume items, and L1 is more robust to these.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / "wow_forecaster.db"
ARTIFACT_DIR = PROJECT_ROOT / "data" / "outputs" / "model_artifacts"
FEATURE_DIR = PROJECT_ROOT / "data" / "processed" / "features"
REALM = "us"

print(f"Project root  : {PROJECT_ROOT}")
print(f"Artifact dir  : {ARTIFACT_DIR}")
print(f"Artifacts exist: {ARTIFACT_DIR.exists()}")

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from wow_forecaster.viz.theme import (
    FEATURE_GROUP_COLORS,
    HORIZON_COLORS,
    WOW_PALETTE,
    apply_wow_theme,
)
from wow_forecaster.viz.data_queries import (
    fetch_archetypes,
    fetch_feature_importance,
    fetch_forecast_data,
    fetch_historical_prices,
)
from wow_forecaster.viz.charts.feature_chart import (
    plot_feature_importance,
    plot_importance_by_horizon,
    plot_feature_correlation,
)
from wow_forecaster.viz.charts.forecast_chart import (
    plot_forecast_timeline,
    plot_forecast_multi_horizon,
)

warnings.filterwarnings("ignore", category=FutureWarning)

fig_tmp, _ = plt.subplots()
apply_wow_theme(fig_tmp)
plt.close(fig_tmp)

print("Imports OK.")

---
## 1. Feature Engineering

The feature registry defines **48 columns** organized into **10 groups**. Each feature is designed with a specific forecasting rationale:

| Group | Count | Purpose |
|-------|-------|---------|
| **price** | 9 | Core OHLC-style daily aggregates (mean, min, max, market_value, historical_value, obs_count) plus identifiers |
| **volume** | 3 | Market depth signals: quantity_sum, auctions_sum, and is_volume_proxy flag |
| **lag** | 5 | Lookback prices at 1d, 3d, 7d, 14d, 28d -- the model's "memory" of recent price levels |
| **rolling** | 6 | Rolling mean and std over 7d, 14d, 28d windows -- captures trend and volatility |
| **momentum** | 3 | Percentage price changes over 7d, 14d, 28d -- direction and acceleration |
| **temporal** | 4 | Calendar features: day_of_week, day_of_month, week_of_year, days_since_expansion |
| **event** | 8 | Event proximity, severity, archetype impact, pre-event window flag -- all leakage-safe |
| **archetype** | 5 | Category encoding, transferability flag, cold-start indicator, item count |
| **transfer** | 2 | Cross-expansion mapping flag and confidence score |
| **target** | 3 | Forward-looking labels: target_price_1d, target_price_7d, target_price_28d (training only) |

### Feature group rationale

**Why lag + rolling + momentum?** These three groups capture different facets of the same signal:
- Lags tell the model "what was the price N days ago?" (level)
- Rolling stats tell it "what is the recent trend and volatility?" (distributional)
- Momentum tells it "is the price accelerating up or down?" (derivative)

LightGBM's tree splits can combine these non-linearly -- e.g., "if momentum is positive AND volatility is low AND an event is approaching, predict a further price increase."

**Why event features?** In-game events are the most predictable external shocks. The pre-event window (7 days before a major event) consistently shows demand-driven price increases for consumables. The leakage guard (`announced_at <= obs_date`) ensures the model only uses information that was publicly available at prediction time.

In [ ]:
# ── Feature registry overview ────────────────────────────────────────────────

from wow_forecaster.features.registry import (
    FEATURE_REGISTRY,
    feature_groups,
    feature_names,
    training_feature_names,
    inference_feature_names,
    target_feature_names,
)

print(f"Total features in registry  : {len(FEATURE_REGISTRY)}")
print(f"Training columns            : {len(training_feature_names())}")
print(f"Inference columns           : {len(inference_feature_names())}")
print(f"Target columns (train only) : {len(target_feature_names())}")
print(f"Feature groups              : {feature_groups()}")
print()

# Group breakdown
for group in feature_groups():
    names = feature_names(group)
    print(f"  {group:12s} ({len(names):2d}): {', '.join(names)}")

In [ ]:
# ── Feature details table ────────────────────────────────────────────────────

feature_data = []
for spec in FEATURE_REGISTRY:
    feature_data.append({
        "name": spec.name,
        "type": spec.pa_type,
        "group": spec.group,
        "nullable": spec.is_nullable,
        "min_history_days": spec.requires_history_days,
        "is_target": spec.is_target,
        "description": spec.description[:80] + "..." if len(spec.description) > 80 else spec.description,
    })

df_features = pd.DataFrame(feature_data)
# Show features that require history (cold-start considerations)
history_features = df_features[df_features["min_history_days"] > 0]
print(f"Features requiring prior history: {len(history_features)}")
print(history_features[["name", "group", "min_history_days"]].to_string(index=False))

The history requirements matter for cold-start scenarios. A brand-new Midnight archetype needs **28 days** of observations before the longest lag/rolling features become non-null. LightGBM handles the intermediate nulls natively (NaN propagation), but the model is less confident with sparse inputs -- this feeds into the uncertainty penalty.

---
## 2. Feature Importance Analysis

LightGBM provides two importance metrics:

- **Gain**: Total reduction in the loss function contributed by all splits on this feature. Features with high gain are the most informative for prediction accuracy.
- **Split count**: Number of times the feature was used in a tree split. Features with high split counts are used frequently but each individual split may contribute less.

We focus on **gain** as it better reflects predictive value.

In [ ]:
# ── Load feature importance from model artifacts ─────────────────────────────

df_importance = fetch_feature_importance(ARTIFACT_DIR, REALM)

if not df_importance.empty:
    print(f"Feature importance records loaded: {len(df_importance)}")
    print(f"Horizons available: {sorted(df_importance['horizon'].unique())}")
else:
    print(
        "No model artifacts found. Run `wow-forecaster run-daily-forecast` to train models.\n"
        f"Expected location: {ARTIFACT_DIR}"
    )

In [ ]:
# ── Top-15 features by gain importance ───────────────────────────────────────

if not df_importance.empty:
    fig = plot_feature_importance(df_importance, top_n=15, importance_type="gain")
    plt.show()
else:
    print("Skipping feature importance chart -- no model artifacts.")

In [ ]:
# ── Feature importance by horizon (1d / 7d / 28d) ───────────────────────────

if not df_importance.empty:
    fig = plot_importance_by_horizon(df_importance, top_n=10)
    plt.show()
else:
    print("Skipping horizon comparison -- no model artifacts.")

**Expected importance patterns:**

- For the **1d horizon**, short-term features dominate: `price_lag_1d`, `price_pct_change_7d`, and `price_mean` -- the recent price is the best predictor of tomorrow's price.
- For the **7d horizon**, rolling statistics gain importance: `price_roll_mean_7d`, `price_roll_std_7d` -- the model shifts from point prediction to trend following.
- For the **28d horizon**, structural features become more relevant: `days_since_expansion`, `archetype_category_enc`, event features -- longer horizons require understanding the macro context.

This horizon-dependent feature shift validates our multi-horizon design: a single model with a fixed feature set cannot optimally serve all horizons.

---
## 3. Feature Correlation

Understanding feature correlations helps identify redundancy and multicollinearity. While LightGBM is generally robust to correlated features (tree splits are independent), high correlations can indicate opportunities to simplify the feature set.

We expect strong correlations between:
- Lag features at similar windows (lag_7d and lag_14d)
- Rolling means at different windows (roll_mean_7d and roll_mean_14d)
- price_mean and price_lag_1d (consecutive days are highly correlated)

These correlations are acceptable because tree-based models handle them gracefully via feature selection at each split.

In [ ]:
# ── Load training Parquet for correlation analysis ───────────────────────────

training_parquet = None
if FEATURE_DIR.exists():
    # Find the latest training Parquet
    parquet_files = sorted(FEATURE_DIR.glob("training_*.parquet"), reverse=True)
    if parquet_files:
        training_parquet = parquet_files[0]
        print(f"Loading training Parquet: {training_parquet.name}")

if training_parquet and training_parquet.exists():
    df_train = pd.read_parquet(training_parquet)
    print(f"Shape: {df_train.shape}")
    print(f"Columns: {df_train.columns.tolist()[:10]}...")
else:
    df_train = pd.DataFrame()
    print(
        "No training Parquet found. Run `wow-forecaster build-datasets` first.\n"
        f"Expected location: {FEATURE_DIR}"
    )

In [ ]:
# ── Feature correlation heatmap (top 20 by variance) ─────────────────────────

if not df_train.empty:
    fig = plot_feature_correlation(df_train, top_n=20)
    plt.show()
else:
    print("Skipping correlation heatmap -- no training data loaded.")

In [ ]:
# ── Feature variance analysis ────────────────────────────────────────────────
# Low-variance features contribute little predictive signal.

if not df_train.empty:
    numeric_cols = df_train.select_dtypes(include=[np.number])
    variances = numeric_cols.var().sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    apply_wow_theme(fig)

    # Show bottom 15 (lowest variance)
    bottom = variances.head(15)
    colors = [
        FEATURE_GROUP_COLORS.get(
            next(
                (spec.group for spec in FEATURE_REGISTRY if spec.name == f),
                "price"
            ),
            WOW_PALETTE["primary"]
        )
        for f in bottom.index
    ]

    ax.barh(bottom.index, np.log10(bottom.values.clip(min=1e-10)),
            color=colors, alpha=0.8,
            edgecolor=WOW_PALETTE["grid"], linewidth=0.5)
    ax.set_xlabel("log10(Variance)")
    ax.set_title("Lowest-Variance Features (log scale)", fontweight="bold")
    fig.tight_layout()
    plt.show()
else:
    print("Skipping variance analysis -- no training data.")

---
## 4. Why LightGBM?

The choice of LightGBM over alternatives is grounded in the specific constraints of this problem:

### vs. Prophet / ARIMA (time-series models)
- Prophet needs 30+ observations per series for seasonality estimation. Cold-start Midnight archetypes may have fewer than 14 days of data.
- Classical time-series models cannot consume **cross-series features** (event severity, archetype category, transfer confidence) that distinguish different market segments.
- A global LightGBM model trains across all series, sharing knowledge.

### vs. XGBoost
- LightGBM's histogram-based leaf-wise growth is faster and uses less memory.
- Accuracy differences on tabular data are marginal.

### vs. LSTM / Transformer
- Deep sequence models need 100+ timesteps per series. Cold-start archetypes don't have this.
- Inference is orders of magnitude slower.
- Feature interactions (events + category + price momentum) are naturally captured by tree splits without explicit architecture design.

### Training strategy

One global model per horizon, trained on all archetype-realm series simultaneously:

```
                    +-----------+
  All archetypes -->| LightGBM  |---> 1d price forecast
  All realms    -->| (global)  |---> (per archetype)
  All dates     -->|           |
                    +-----------+
```

The model receives `archetype_category_enc` and `is_cold_start_int` as features, enabling it to learn category-specific patterns without memorizing individual archetype IDs.

In [ ]:
# ── Model hyperparameters ────────────────────────────────────────────────────

from wow_forecaster.ml.lgbm_model import LightGBMForecaster
from wow_forecaster.ml.feature_selector import (
    TRAINING_FEATURE_COLS,
    CATEGORICAL_FEATURE_COLS,
    TARGET_COL_MAP,
)

print("Model configuration:")
print(f"  Model features (input)    : {len(TRAINING_FEATURE_COLS)} columns")
print(f"  Categorical features      : {CATEGORICAL_FEATURE_COLS}")
print(f"  Target columns            : {TARGET_COL_MAP}")
print(f"  Model artifact version    : {LightGBMForecaster.MODEL_VERSION}")
print()

# Show default hyperparameters
default_model = LightGBMForecaster(horizon_days=1)
print("Default hyperparameters:")
for key, val in default_model._hyperparams.items():
    print(f"  {key:20s}: {val}")
print(f"  {'early_stopping':20s}: {default_model._early_stopping_rounds} rounds")
print(f"  {'objective':20s}: regression_l1 (MAE loss)")

---
## 5. Hyperparameter Discussion

The current hyperparameters represent a conservative starting point:

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `num_leaves` | 31 | Standard default; enough capacity without overfitting on thin series |
| `learning_rate` | 0.05 | Moderate -- pairs with 100 estimators for reasonable convergence |
| `n_estimators` | 100 | With early stopping at 20 rounds, typically stops around 60-80 |
| `min_child_samples` | 5 | Low to accommodate cold-start archetypes with few rows; **top tuning priority** |
| `feature_fraction` | 0.8 | Column subsampling for regularization |
| `bagging_fraction` | 0.8 | Row subsampling for regularization |
| `bagging_freq` | 5 | Apply bagging every 5 iterations |
| `objective` | regression_l1 | MAE loss -- robust to price outliers |

### Deferred tuning priorities

1. **`min_child_samples`** (5 -> 15-30): Currently very low, which risks overfitting on thin cold-start series. Needs LightGBM backtest integration first.
2. **`learning_rate`** (0.05 -> 0.02-0.03): Lower rate with more estimators for smoother convergence.
3. **`num_leaves`** (31 -> 20-50): Search for optimal tree complexity.

Tuning requires a proper LightGBM backtest harness (architecturally different from the baseline evaluator) and 60+ days of Midnight data for 1d/7d horizons.

In [ ]:
# ── Load trained model artifacts and show validation metrics ─────────────────

if ARTIFACT_DIR.exists():
    model_files = sorted(ARTIFACT_DIR.glob(f"lgbm_*_{REALM}_*.pkl"), reverse=True)
    if model_files:
        print(f"Found {len(model_files)} model artifact(s):\n")
        seen_horizons = set()
        for mf in model_files:
            try:
                model = LightGBMForecaster.load(mf)
                h = f"{model.horizon_days}d"
                if h in seen_horizons:
                    continue
                seen_horizons.add(h)
                metrics = model.val_metrics
                print(f"Horizon: {h}")
                print(f"  Artifact      : {mf.name}")
                print(f"  Training rows : {model._training_rows:,}")
                print(f"  Trained at    : {model._trained_at}")
                if metrics:
                    for k, v in metrics.items():
                        print(f"  Val {k:5s}     : {v:.4f}")
                else:
                    print("  (no validation metrics -- trained without val set)")
                print()
            except Exception as exc:
                print(f"  Failed to load {mf.name}: {exc}")
    else:
        print(f"No model artifacts found in {ARTIFACT_DIR}")
else:
    print(f"Artifact directory does not exist: {ARTIFACT_DIR}")

---
## 6. Forecast Examples

Let's look at actual forecasts produced by the model. Each forecast includes:

- **Point prediction**: Expected price at the target horizon
- **Confidence interval**: Lower and upper bounds
- **CI quality**: "good" (narrow), "wide", or "unreliable" (based on floor/cap rules)
- **Model slug**: Which model produced the forecast (includes `_transfer` suffix for cold-start blends)

The confidence interval is currently heuristic-based (not quantile regression). It uses:
- Base width from rolling standard deviation
- Widened by uncertainty multiplier from drift detection
- Floor at 5% of current price, cap at 10x current price
- Cold-start archetypes get an additional 1.5x multiplier

In [ ]:
# ── Fetch forecast data ──────────────────────────────────────────────────────

df_forecasts = fetch_forecast_data(DB_PATH, REALM)
df_archetypes = fetch_archetypes(DB_PATH)

if not df_forecasts.empty:
    print(f"Total forecasts: {len(df_forecasts)}")
    print(f"Horizons: {sorted(df_forecasts['forecast_horizon'].unique())}")
    if 'ci_quality' in df_forecasts.columns:
        print(f"\nCI quality distribution:")
        print(df_forecasts['ci_quality'].value_counts().to_string())
    if 'model_slug' in df_forecasts.columns:
        print(f"\nModel slugs:")
        print(df_forecasts['model_slug'].value_counts().head(10).to_string())
else:
    print("No forecast data. Run `wow-forecaster run-daily-forecast` first.")

In [ ]:
# ── Forecast timeline for selected archetypes ────────────────────────────────
# Show historical price + forecast point + CI band for a few archetypes.

if not df_forecasts.empty and not df_archetypes.empty:
    # Pick up to 3 archetypes with forecasts and sufficient history
    archetype_forecasts = df_forecasts[
        df_forecasts["archetype_id"].notna()
    ]["archetype_id"].unique()

    shown = 0
    for arch_id in archetype_forecasts[:10]:  # check first 10
        if shown >= 3:
            break

        arch_id_int = int(arch_id)
        arch_meta = df_archetypes[df_archetypes["archetype_id"] == arch_id_int]
        if arch_meta.empty:
            continue
        arch_name = arch_meta.iloc[0].get("display_name", f"Archetype {arch_id_int}")

        df_hist = fetch_historical_prices(DB_PATH, REALM, arch_id_int, days=60)
        if df_hist.empty or len(df_hist) < 7:
            continue

        arch_fc = df_forecasts[df_forecasts["archetype_id"] == arch_id]

        # Multi-horizon view
        fig = plot_forecast_multi_horizon(df_hist, arch_fc, archetype_name=arch_name)
        plt.show()
        shown += 1

    if shown == 0:
        print("No archetypes with both history and forecasts found.")
else:
    print("No forecast or archetype data available.")

In [ ]:
# ── CI quality by horizon ────────────────────────────────────────────────────

if not df_forecasts.empty and 'ci_quality' in df_forecasts.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    apply_wow_theme(fig)

    horizons = sorted(df_forecasts['forecast_horizon'].unique())
    qualities = ['good', 'wide', 'unreliable']
    quality_colors = {'good': '#2ECC71', 'wide': '#F39C12', 'unreliable': '#E74C3C'}

    x = np.arange(len(horizons))
    width = 0.25

    for i, q in enumerate(qualities):
        counts = []
        for h in horizons:
            mask = (df_forecasts['forecast_horizon'] == h) & (df_forecasts['ci_quality'] == q)
            counts.append(mask.sum())
        ax.bar(x + i * width, counts, width, label=q.title(),
               color=quality_colors.get(q, WOW_PALETTE['primary']), alpha=0.8)

    ax.set_xticks(x + width)
    ax.set_xticklabels(horizons)
    ax.set_xlabel('Forecast Horizon')
    ax.set_ylabel('Count')
    ax.set_title('CI Quality Distribution by Horizon', fontweight='bold')
    ax.legend(framealpha=0.8)
    fig.tight_layout()
    plt.show()
else:
    print("No CI quality data for visualization.")

**Key observations:**

- The 1d horizon typically has the tightest CIs (short prediction window = less uncertainty).
- The 28d horizon naturally produces wider CIs -- this is appropriate, not a model failure.
- Cold-start (`_transfer` suffix) models have wider CIs by design, reflecting genuine uncertainty.
- The CI floor/cap mechanism prevents pathological cases: no CI lower bound below 5% of current price, no upper bound above 10x.

---

**Next: Notebook 03** dives into formal backtesting -- walk-forward validation, metric breakdown by category, calibration analysis, and the recommendation scoring trace.